## CarDekho Used Car Dataset — Data Cleaning & Preprocessing
---
### Objective

This notebook performs **end-to-end data cleaning and preprocessing** on the CarDekho Used Cars Dataset. The cleaned dataset will be used to build a **Multiple Linear Regression (MLR)** model that predicts the **listed price** of a used car.

---

### About the Dataset

| Property | Details |
|-|-|
| **Rows** | 37,813 |
| **Columns (raw)** | 66 features |
| **Target variable** | `listed_price` — the price of the car (₹)|

The features cover vehicle specifications (engine, dimensions, power), ownership history, location, fuel type, transmission, and more.

---

### What This Notebook Covers

1. Import libraries and load dataset  
2. Explore dataset structure  
3. Feature selection (66 → 19 columns)  
4. Standardize column names  
5. Fix data type inconsistencies  
6. Handle missing values  
7. Detect and treat outliers  
8. Feature engineering  
9. Encode categorical variables  
10. Save cleaned dataset  

---

### Why Is Data Cleaning Important?

Raw real-world datasets almost always have problems:

- **Missing values** — models cannot train on gaps in the data
- **Wrong data types** — e.g. a price stored as text instead of a number
- **Outliers** — extreme values that skew the model and inflate errors
- **Redundant or irrelevant columns** — noise that degrades performance
- **Inconsistent formatting** — e.g. column names with mixed casing

Skipping this step leads to unreliable models, misleading insights, and violated statistical assumptions. **Data cleaning is the foundation of every good data science project.**


---
### Step 1 — Import Libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import math

import warnings
warnings.filterwarnings('ignore')

---

### Step 2 — Load the Dataset

In [2]:
dataset = pd.read_csv("datasets/Car_Dekho_Dataset.csv")
df = pd.DataFrame(dataset)
print("Shape of the dataset: ", df.shape)
print()
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/goyalaana/car-dekho-dataset/Car_Dekho_Dataset.csv'

### Dataset Structure (`df.info()`)

`df.info()` gives us a concise summary of the entire DataFrame:

- **Column names** and their **data types** 
- **Non-null count** per column
- **Memory footprint** of the DataFrame

This output is our starting point for identifying what needs to be cleaned.

In [ ]:
df.info()

### Key Observations

Based on `df.info()`, the following insights can be drawn:

- The dataset contains **37,813 rows and 66 columns**, indicating a high-dimensional structure.  
- Several columns have **missing values**, with some features having very low non-null counts (e.g., `Stroke`, `Bore`, `Ground Clearance Unladen`).  
- The dataset includes a mix of **float, integer, object, and boolean** data types, requiring preprocessing.  
- Some columns contain **complex or grouped information** (e.g., `top_features`, `safety_features`) stored as objects.  
- Certain columns (e.g., IDs, images, highly sparse technical features) may be **irrelevant for analysis** and need evaluation.

---

### Step 3 — Feature Selection
The original 66 columns contain many features that are either:
- Mostly **empty** (e.g. `Stroke` — only 513 out of 37 813 rows filled in)
- **Not useful** for predicting price (e.g. `images`, `usedCarSkuId`)
- **Redundant** (e.g. both `loc` and `City` represent location)

We manually select **19 columns** that are most likely to influence the listed price.

> **Why these features?**  
> Domain knowledge tells us that price is driven by: the car's age, brand/model, how much it's been driven, engine power, physical size, where it's being sold, and ownership history.

In [ ]:
important_cols = [
    "myear",'km','model','fuel','transmission','owner_type','Seats',
    'Drive Type','Engine Type','No of Cylinder','Max Power Delivered',
    'Max Torque Delivered','Length','Width','Height','Kerb Weight',
    'Gear Box','City','listed_price'
]

df = df[important_cols]
df.head()

---

### Step 4 — Standardise Column Names

After this step, all column names follow a clean **`Title Case with Spaces`** convention, which looks professional in output tables and is easy to reference in code.


In [ ]:
df.columns = df.columns.str.replace("_",' ')
df.columns = df.columns.str.title()
df.columns

---

### Step 5 — Data Type Conversion

Fixing data types is essential before any analysis or modelling. Wrong types lead to errors, unexpected behaviour, and missed optimisations.
1. Columns like `Seats`, `No Of Cylinder`, and `Listed Price` should be **whole numbers** (we can't have 4.5 cylinders).  
2. The `Gear Box` column contains very messy free-text values like `'5 speed automatic'`, `'6 speed cvt'`, etc.  

In [ ]:
float_cols_conversion = df[["Seats","No Of Cylinder","Listed Price"]]
for col in float_cols_conversion:
    df[col] = df[col].astype("Int64")

In [ ]:
def classify_gear(x):
    if pd.isna(x):
        return pd.NA
    x = x.lower()
    if 'cvt' in x:
        return 'CVT'
    elif 'automatic' in x:
        return 'Automatic'
    elif 'direct' in x:
        return 'Direct'
    else:
        return 'Manual'

df['Gear Type'] = df['Gear Box'].apply(classify_gear)
df.drop("Gear Box",axis=1,inplace=True)

In [ ]:
df.info()

---

### Step 6 — Missing Value Analysis & Imputation

#### Why Handle Missing Values?
- Most ML algorithms **cannot process `NaN` values** and will throw errors
- Dropping rows with many missing value would remove too many samples
- Imputing incorrectly (e.g. using 0 for all gaps) would **introduce bias**

In [ ]:
df.isnull().sum() # Count of missing values per column

#### Visualise Distributions (Histograms)

Before choosing how to fill gaps, we plot histograms of all numerical columns to understand their **shape and spread**. This tells us:
- Is the distribution **symmetric** (use mean) or **skewed** (use median)?

In [ ]:
num_cols = df.select_dtypes(include=["float64", "int64"]).columns
df[num_cols].hist(figsize=(15,10), bins=30)
plt.tight_layout()
plt.show()

#### Histogram Observations

Based on the distributions above:

- **`Km` (kilometres driven)** — strongly right-skewed; most cars have low mileage but a few have extremely high values → use **median imputation** and will need **outlier treatment** later.
- **`Listed Price`** — heavily right-skewed; most cars are budget-priced but luxury outliers pull the tail far right → **median imputation** + outlier clipping needed.
- **`Max Power Delivered` / `Max Torque Delivered`** — roughly right-skewed; a small number of high-performance cars have much higher values → **median imputation** is appropriate.
- **`Seats`**, **`No Of Cylinder`** — discrete distributions with clear peaks; **mode imputation** is sensible.
- **`Length`, `Width`, `Height`** — reasonably symmetric within each car model → **group mean** is a good fit.


In [ ]:
numerical_cols = ["No Of Cylinder", "Max Power Delivered", "Max Torque Delivered"]
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

Int64_cols_conversion = df[["No Of Cylinder","Listed Price"]]
for col in Int64_cols_conversion:
    df[col] = df[col].astype("int64")

In [ ]:
categorical_cols = ["Drive Type","Gear Type","Seats"]
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
df["Seats"] = df["Seats"].astype("int64")

In [ ]:
for col in ["Length", "Height", "Width"]:
    df[col] = df.groupby("Model")[col].transform(lambda x: x.fillna(x.mean()))
    df[col] = df[col].fillna(df[col].mean())  # fallback

In [ ]:
df['Engine Type'] = df.groupby('Model')['Engine Type'] \
                      .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

df['Engine Type'] = df['Engine Type'].fillna(df['Engine Type'].mode()[0])

In [ ]:
df.drop("Kerb Weight",axis=1, inplace=True) ## Kerb Weight column has a lot of missing values so filling them would add bias to the model.

In [ ]:
df.info()

---
### Step 7 — Outlier Detection & Treatment

#### What Is an Outlier?
An outlier is a data point that lies **far outside the typical range** of a column. Outliers cause problems because:
- They **skew the model** — regression lines get pulled towards extreme values
- They **inflate error metrics** — one bad prediction can ruin our RMSE
- They often represent **data entry errors** or genuinely rare edge cases
- 
### Detection — Box Plots
We plot a **box plot** for every numerical column. Box plots compactly show:
- The **median** (centre line)
- The **interquartile range – IQR** (the box, covering the middle 50 % of data)
- **Whiskers** at 1.5 × IQR above/below the box
- Individual **outlier dots** beyond the whiskers

In [ ]:
num_cols = df.select_dtypes(include=["float64", "int64"]).columns

n = len(num_cols)

for i in range(0, n, 3):
    plt.figure(figsize=(15, 3))
    
    for j in range(3):
        if i + j < n:
            col = num_cols[i + j]
            plt.subplot(1, 3, j + 1)
            sns.boxplot(x=df[col])
            plt.title(col)
    
    plt.tight_layout()
    plt.show()

#### Outlier Observations

From the box plots:

- **`Seats`, `No Of Cylinder`, `Car Age`** — outliers exist but are **reasonable and valid**. A 12-cylinder car or a 9-seater MPV is genuinely unusual, but real. We leave these alone.
- **`Km`** — some cars show 300,000+ km. These could be real, but they are extreme and will distort the model. → **Clip**
- **`Max Power Delivered` / `Max Torque Delivered`** — a handful of supercars (Lamborghini, Ferrari) have extreme values that would skew the regression. → **Clip**
- **`Listed Price`** — a few cars priced above ₹1 Crore are genuine luxury vehicles, but they are outliers relative to the typical used-car market. → **Clip**

#### Treatment — IQR Clipping
Any value **below the lower fence** is set equal to the lower fence.  
Any value **above the upper fence** is set equal to the upper fence.

> **Why clip instead of delete?**  
> Deleting outlier rows removes valid data and reduces dataset size. Clipping keeps every row but **caps** extreme values at a reasonable limit — the best of both worlds.

In [ ]:
num_cols = ['Km', 'Max Power Delivered', 'Max Torque Delivered', 'Listed Price']

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

In [ ]:
num_cols = df.select_dtypes(include=["float64", "int64"]).columns

n = len(num_cols)

for i in range(0, n, 3):
    plt.figure(figsize=(15, 3))
    
    for j in range(3):
        if i + j < n:
            col = num_cols[i + j]
            plt.subplot(1, 3, j + 1)
            sns.boxplot(x=df[col])
            plt.title(col)
    
    plt.tight_layout()
    plt.show()

---

### Step 8 — Feature Engineering
**Feature engineering** means transforming or combining existing columns to create **new, more informative features**.

### Why Feature Engineering?

- A model can't use free-text like `'5 speed automatic'` — we need numbers
- `Myear = 2018` tells the model less than `Car Age = 8`
- Three separate dimension columns (Length, Width, Height) can become one `Car Size` volume
- Extracting information buried in a text field (e.g. whether an engine is turbocharged) adds signal

In [ ]:
## A 'car's age' has a more direct, intuitive relationship with price than the calendar year does.

df["Car Age"] = 2026 - df["Myear"]
df.drop("Myear",axis=1,inplace=True)


## Kilometers driven is a key indicator of usage. To simplify analysis, it is grouped into categories: Low, Medium, and High usage.
def usage_type(x):
    if(x < 20000):
        return "Low"
    elif (x < 80000):
        return "Median"
    else:
        return "High"
df["Usage Type"] = df["Km"].apply(usage_type)

In [ ]:
df["Model"].unique()

In [ ]:
df["Brand"] = df["Model"].str.split().str[0]

In [ ]:
## Brands are grouped into different price tiers based on their market positioning to capture brand value more effectively.
def segment(x):
    x = x.lower()
    if any(k in x for k in ['swift', 'i10', 'alto', 'wagon', 'polo', 'kwid']):
        return 'Hatchback'
    elif any(k in x for k in ['city', 'verna', 'ciaz', 'amaze']):
        return 'Sedan'
    elif any(k in x for k in ['creta', 'scorpio', 'fortuner', 'xuv', 'harrier', 'seltos']):
        return 'SUV'
    else:
        return 'Other'

df['Car Segment'] = df['Model'].apply(segment)

In [ ]:
df.drop("Model",axis=1,inplace=True)

In [ ]:
df["Car Size"] = (df["Length"]*df["Width"]*df["Height"])/1e9
df.drop(columns = ["Length","Width","Height"],inplace=True)

In [ ]:
df["Engine Type"].unique()

In [ ]:
## A binary feature indicating whether the engine is turbocharged, as turbo engines often increase performance and price.
df['Is Turbo'] = df['Engine Type'].str.lower().apply(lambda x: 1 if 'turbo' in str(x) else 0)

In [ ]:
## A binary feature capturing advanced engine technologies (e.g., DOHC, VVT, TSI), which are typically associated with better performance.
df['Is Advanced Engine'] = df['Engine Type'].str.lower().apply(
    lambda x: 1 if any(k in str(x) for k in ['dohc', 'vvt', 'tsi', 'tfsi', 'gdi']) else 0
)

In [ ]:
df.drop("Engine Type",axis = 1, inplace = True)

In [ ]:
df.info()

---

### Step 9 — Encoding Categorical Variables

Machine learning models work with **numbers**, not text. Every categorical column must be converted into a numerical representation.

We have **9 categorical columns** after feature engineering. The right encoding strategy depends on the column's characteristics.

In [ ]:
cols = df.select_dtypes(include = "object")
for col in cols:
    print(col, df[col].nunique())

#### - Frequency Encoding → `City`

`City` has **617 unique values** — way too many for one-hot encoding (that would add 617 new columns!).

Instead, we replace each city name with the **count of listings from that city**:
```
Mumbai   → 2,450   (very popular market)
Lucknow  →   890
Jasola   →    12   (small market)
```
This is called **Frequency Encoding**. It captures a useful signal: cities with more listings tend to be bigger, more liquid markets — which can affect both supply and pricing.

In [ ]:
city_counts = df["City"].value_counts()   
df["City Freq"] = df["City"].map(city_counts)
df.drop(columns=["City"],inplace=True)

#### - Brand Grouping + One-Hot Encoding → `Brand`

With **44 unique brands**, one-hot encoding directly would add 44 columns — many with very few rows. Instead, we first **group brands into 7 price tiers** based on market positioning.

This `Brand Category` column (7 levels) is then **one-hot encoded**.

In [ ]:
def brand_category(x):
    x = x.lower()
    if x in ['maruti', 'hyundai', 'tata', 'datsun', 'renault']:
        return 'Budget'
    
    elif x in ['honda', 'toyota', 'kia', 'nissan', 'volkswagen', 'skoda', 'ford']:
        return 'Mid'
    
    elif x in ['jeep', 'mg', 'citroen', 'fiat', 'isuzu']:
        return 'Premium'
    
    elif x in ['bmw', 'audi', 'mercedes-benz', 'jaguar', 'volvo', 'lexus']:
        return 'Luxury'
    
    elif x in ['porsche', 'ferrari', 'lamborghini', 'bentley', 'aston', 'rolls-royce', 'maserati']:
        return 'Ultra_Luxury'
    
    elif x in ['mahindra', 'force', 'ashok', 'icml']:
        return 'Commercial'
    
    else:
        return 'Other'
df['Brand Category'] = df['Brand'].apply(brand_category)
df.drop(columns = ['Brand'],inplace = True)

#### - One-Hot Encoding → Remaining Low-Cardinality Columns

These columns have few unique values and no natural order, so one-hot encoding is ideal.

In [ ]:
one_hot_columns = ["Brand Category","Fuel","Car Segment","Transmission","Drive Type","Gear Type"]
df = pd.get_dummies(df,columns = one_hot_columns,drop_first=True) ## drop_first -->> To avoid multicolinearity.

bool_cols = df.select_dtypes(include="bool")
for col in bool_cols:
    df[col] = df[col].apply(lambda x: 1 if 'True' else 0)

In [ ]:
cols = df.select_dtypes(include = "object")
for col in cols:
    print(col, df[col].unique())

#### - Ordinal Encoding → `Owner Type` & `Usage Type`

These two columns have a **meaningful natural order** that carries real information:

We use `sklearn`'s `OrdinalEncoder` to map these to integers (0, 1, 2, …) while **preserving the order**. This is important — using one-hot encoding here would lose the ordering information entirely.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
owner_order = [['unregistered car', 'first', 'second', 'third', 'fourth', 'fifth']]
encoder = OrdinalEncoder(categories=owner_order)
df['Owner Type'] = encoder.fit_transform(df[['Owner Type']])

In [ ]:
usage_order = [['Low', 'Median', 'High']]
encoder = OrdinalEncoder(categories=usage_order)
df['Usage Type'] = encoder.fit_transform(df[['Usage Type']])

In [ ]:
df.info()

### Step 10 - Save the Cleaned Dataset
> 🎉 The dataset is now **clean, consistent, and ready for machine learning!**


In [ ]:
df.to_csv("Cleaned_CarDekho_dataset.csv",index=False)